# ĐỀ TÀI: PHÂN TÍCH VÀ DỰ ĐOÁN CHẤT LƯỢNG KHÔNG KHÍ (AQI) TẠI HÀ NỘI
**Các bước thực hiện:**
1. Khám phá dữ liệu (EDA)
2. Trực quan hóa (Visualization)
3. Xây dựng mô hình học máy (Modeling)
4. Rút ra nhận định (Insights)

In [2]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Thiết lập style cho biểu đồ
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 1. Khám phá dữ liệu (EDA)

In [ ]:
# Đọc dữ liệu đã làm sạch
df = pd.read_csv('data/Du_lieu_Hanoi_Cleaned.csv')

# Xem 5 dòng đầu tiên
display(df.head())

# Kiểm tra thông tin các cột và kiểu dữ liệu
print('\n--- Thông tin dữ liệu ---')
df.info()

# Thống kê mô tả các biến số
print('\n--- Thống kê mô tả ---')
display(df.describe())

FileNotFoundError: [Errno 2] No such file or directory: 'Du_lieu_Hanoi_Cleaned.csv'

## 2. Trực quan hóa dữ liệu (Visualization)

In [ ]:
# 2.1 Biểu đồ phân bố của AQI
plt.figure(figsize=(10, 5))
sns.histplot(df['aqi'], bins=30, kde=True, color='coral')
plt.title('Phân bố Chỉ số chất lượng không khí (AQI) tại Hà Nội')
plt.xlabel('AQI')
plt.ylabel('Tần suất')
plt.show()

In [ ]:
# 2.2 Đánh giá mức độ ô nhiễm theo khung giờ trong ngày
plt.figure(figsize=(12, 6))
sns.boxplot(x='hour', y='aqi', data=df, palette='coolwarm')
plt.title('Mức độ ô nhiễm AQI theo từng khung giờ trong ngày')
plt.xlabel('Giờ trong ngày')
plt.ylabel('AQI')
plt.show()

In [ ]:
# 2.3 Ma trận tương quan giữa các biến số số học
plt.figure(figsize=(8, 6))
correlation_matrix = df[['aqi', 'wind_speed', 'humidity', 'hour', 'month']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', fmt='.2f', vmin=-1, vmax=1)
plt.title('Ma trận tương quan giữa các biến số')
plt.show()

## 3. Xây dựng mô hình Dự đoán (Modeling)
Sử dụng thuật toán **Random Forest Regressor** để dự đoán chỉ số AQI dựa trên các yếu tố khí tượng và thời gian.

In [ ]:
# Chọn các biến độc lập (X) và biến phụ thuộc (y)
features = ['wind_speed', 'humidity', 'hour', 'month']
X = df[features]
y = df['aqi']

# Xử lý các giá trị NaN nếu có
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# Chia tập dữ liệu thành Train (80%) và Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Khởi tạo và huấn luyện mô hình Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Dự đoán trên tập Test
y_pred = model.predict(X_test)

# Đánh giá mô hình
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Đánh giá mô hình Random Forest Regressor:")
print(f"- Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"- R-squared (R2 Score): {r2:.2f}")

In [ ]:
# Biểu đồ so sánh giá trị Thực tế và Dự đoán
plt.figure(figsize=(10, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, color='red')
plt.title('So sánh AQI Thực tế vs Dự đoán')
plt.xlabel('AQI Thực tế')
plt.ylabel('AQI Dự đoán')
plt.show()

## 4. Rút ra nhận định (Insights)

In [ ]:
# Hiển thị mức độ quan trọng của từng đặc trưng (Feature Importance)
feature_importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=feature_importances, y=feature_importances.index, palette='viridis')
plt.title('Mức độ quan trọng của các yếu tố ảnh hưởng đến AQI')
plt.xlabel('Điểm quan trọng (Importance Score)')
plt.ylabel('Đặc trưng (Features)')
plt.show()

print("\n--- TỔNG HỢP NHẬN ĐỊNH (INSIGHTS) ---")
print("1. Xu hướng theo thời gian: Mức độ ô nhiễm có sự biến động lớn giữa các khung giờ trong ngày.")
print("2. Sự tương quan: Độ ẩm và Tốc độ gió là các yếu tố khí tượng ảnh hưởng trực tiếp đến chỉ số AQI.")
print("3. Mô hình dự đoán: Thuật toán Random Forest chỉ ra rằng [Độ ẩm] và [Tốc độ gió] đóng vai trò quan trọng nhất trong việc xây dựng mô hình dự báo mức độ ô nhiễm tại Hà Nội.")